In [26]:

import torch
import torch.nn as nn
import torch.optim as optim
from PIL import Image
from torch.utils.data import DataLoader
from torchvision import transforms,datasets
import wandb
from key import KEY
import torch.nn.functional as F
import os
from sklearn.metrics import confusion_matrix
from tqdm import tqdm

In [27]:
img = Image.open('data/img.jpg')

In [28]:
wandb.login(key=KEY)

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: Appending key for api.wandb.ai to your netrc file: C:\Users\최연호\_netrc


True

In [29]:
def data_load(image_size=64, batch_size=128):
    train_dir = 'data/archive/training_set/training_set'
    val_dir = 'data/archive/validation_set/validation_set'
    test_dir = 'data/archive/test_set/test_set'
    trans= transforms.Compose([transforms.Resize((image_size, image_size)), 
                           transforms.ToTensor(), 
                           transforms.Normalize(mean=[0.485, 0.456, 0.406], 
                           std=[0.229, 0.224, 0.225])])


    train_dataset=datasets.ImageFolder(root=train_dir, transform=trans)
    val_dataset=datasets.ImageFolder(root=val_dir, transform=trans)
    test_dataset=datasets.ImageFolder(root=test_dir, transform=trans)
    train_loader= DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=0)
    val_loader= DataLoader(val_dataset, batch_size=batch_size, shuffle=True, num_workers=0)
    test_loader= DataLoader(test_dataset, batch_size=batch_size, shuffle=True, num_workers=0)

    return train_loader, val_loader, test_loader

In [30]:
class ConvBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super(ConvBlock, self).__init__()
        self.conv = nn.Conv2d(in_channels, out_channels, 3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
    
    def forward(self, x):
        x = F.relu(self.conv(x))
        x = self.pool(x)
        return x

class convolution(nn.Module):
    def __init__(self, hidden_size_list, conv_layer_num, block_num, fc2_out, input_size):
        super(convolution, self).__init__()
        self.conv_layer_num = conv_layer_num
        self.block_num = block_num
        self.input_size = input_size
        
        # Block들을 동적으로 생성
        self.blocks = nn.ModuleList()
        
        # 각 Block당 Conv 레이어 수 계산
        conv_per_block = conv_layer_num // block_num
        
        in_channels = 3
        for block_idx in range(block_num):
            block_layers = nn.ModuleList()
            
            # 각 Block 내부의 Conv 레이어들
            for conv_idx in range(conv_per_block):
                global_conv_idx = block_idx * conv_per_block + conv_idx
                out_channels = hidden_size_list[global_conv_idx]
                # block_layers.append(nn.Conv2d(in_channels, out_channels, 3, padding=1))
                block_layers.append(ConvBlock(in_channels, out_channels))
                in_channels = out_channels
            
            # Block 끝에 Pooling 추가
            block_layers.append(nn.MaxPool2d(2, 2))
            self.blocks.append(block_layers)
        
        # FC 레이어 크기 계산
        self.fc1_size = hidden_size_list[conv_layer_num]
        self.fc2_size = fc2_out
        
        # FC 레이어는 forward에서 동적으로 생성
        self.fc1 = None
        self.fc2 = nn.Linear(self.fc1_size, fc2_out)
    
    def forward(self, x):
        # Block들을 통과
        for block in self.blocks:
            for layer in block:
                if isinstance(layer, nn.Conv2d):
                    x = F.relu(layer(x))
                else:  # MaxPool2d
                    x = layer(x)
        
        # Flatten
        batch_size = x.size(0)
        features = x.size(1) * x.size(2) * x.size(3)
        x = x.view(batch_size, -1)
        
        # FC1 레이어 동적 생성
        if self.fc1 is None:
            self.fc1 = nn.Linear(features, self.fc1_size).to(x.device)
        
        # FC 레이어들 통과
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x
    

In [ ]:
import math
class ResidualConvBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super(ResidualConvBlock, self).__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, 3, padding=1)
        self.conv2 = nn.Conv2d(out_channels, out_channels, 3, padding=1)
        
        # 스킵 커넥션을 위한 1x1 Conv (채널 수가 다를 때)
        self.shortcut = nn.Conv2d(in_channels, out_channels, 1) if in_channels != out_channels else nn.Identity()
        
    def forward(self, x):
        residual = self.shortcut(x)
        
        out = F.relu(self.conv1(x))
        out = self.conv2(out)
        out = F.relu(out)
        out = out + residual
        return out


# 잔차 연결이 포함된 메인 모델
class ResidualConvolution(nn.Module):
    def __init__(self, hidden_size_list, conv_layer_num, block_num, fc2_out, input_size):
        super(ResidualConvolution, self).__init__()
        self.conv_layer_num = conv_layer_num
        self.block_num = block_num
        self.input_size = input_size
        self.start_size= hidden_size_list[0]
        self.max_size= math.log2(hidden_size_list[-1]/hidden_size_list[0])
        # Block들을 동적으로 생성
        self.blocks = nn.ModuleList()
        
        # 각 Block당 Conv 레이어 수 계산
        conv_per_block = conv_layer_num // block_num
        
        in_channels = 3
        for block_idx in range(block_num):
            block_layers = nn.ModuleList()
            
            # 각 Block 내부의 Residual Conv 레이어들
            for conv_idx in range(conv_per_block):
                global_conv_idx = block_idx * conv_per_block + conv_idx
                conv_size = global_conv_idx if global_conv_idx < self.max_size else self.max_size
                
                output_size = self.start_size * (2 ** conv_size) 
                
                block_layers.append(ResidualConvBlock(in_channels, output_size))
                in_channels = output_size
            
            # Block 끝에 Pooling 추가
            block_layers.append(nn.MaxPool2d(2, 2))
            self.blocks.append(block_layers)
        
        # FC 레이어 크기 계산
        self.fc1_size = hidden_size_list[-1]
        self.fc2_size = fc2_out
        
        # FC 레이어는 forward에서 동적으로 생성
        self.fc1 = None
        self.fc2 = nn.Linear(self.fc1_size, fc2_out)
    
    def forward(self, x):
        # Block들을 통과
        for block in self.blocks:
            for layer in block:
                if isinstance(layer, ResidualConvBlock):
                    x = layer(x)
                else:  # MaxPool2d
                    x = layer(x)
        
        # Flatten
        batch_size = x.size(0)
        features = x.size(1) * x.size(2) * x.size(3)
        x = x.view(batch_size, -1)
        
        # FC1 레이어 동적 생성
        if self.fc1 is None:
            self.fc1 = nn.Linear(features, self.fc1_size).to(x.device)
        
        # FC 레이어들 통과
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x

In [31]:
def train_model(model,device, train_loader, val_loader, criterion, optimizer, epochs=10):
    for epoch in range(epochs):
        print(f'Starting Epoch: {epoch+1}...')
        running_loss = 0.0
        model.train()
        model.to(device)

        for i, data in enumerate(train_loader, 0):
            inputs, labels = data
            
            inputs = inputs.to(device)
            labels = labels.to(device)

            optimizer.zero_grad()

            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
        

        model.eval()
        with torch.no_grad():
            for i, data in enumerate(val_loader, 0):
                inputs, labels = data
                inputs = inputs.to(device)
                labels = labels.to(device)

                outputs = model(inputs)
                loss = criterion(outputs, labels)
                running_loss += loss.item()
    return model

In [33]:
def evaluate_model(model,device, test_loader):
    model.eval()
    model.to(device)
    correct = 0
    total = 0
    all_labels = []
    all_preds = []
    with torch.no_grad():
        for images, labels in test_loader:
            images = images.to(device)
            labels = labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
            # 예측값과 실제값 저장
            all_labels.extend(labels.cpu().numpy())
            all_preds.extend(predicted.cpu().numpy())
    accuracy = 100 * correct / total
    conf_matrix = confusion_matrix(all_labels, all_preds)

    return accuracy, conf_matrix

In [34]:
def process_model(selected_model=ResidualConvolution,learning_rate=0.001, image_size=64, batch_size=128,
                   hidden_size_list=[32,128],
                   block_num=2, 
                   conv_layer_num=2, 
                   epochs=10, 
                   i=0,
                   ):
    def log_confusion_matrix(conf_matrix, class_names=['Cat', 'Dog']):
        data = []
        for i in range(len(class_names)):
            for j in range(len(class_names)):
                data.append([class_names[i], class_names[j], conf_matrix[i][j]])

        table = wandb.Table(columns=["Actual", "Predicted", "Count"], data=data)
        wandb.log({"confusion_matrix_table": table})
    #데이터 호출

    wandb.init(project="cnn_test_img_265", name=f"cnn_{i}",config={
        "model": selected_model.__name__,
        "learning_rate": learning_rate, 
        "image_size": image_size,
        "batch_size": batch_size, 
        "epochs": epochs,
        "block_num": block_num,
        "conv_layer_num": conv_layer_num,
        "hidden_size_list": hidden_size_list,
        })
    device= torch.device("cuda" if torch.cuda.is_available() else "cpu")


    train_loader, val_loader, test_loader= data_load(image_size, batch_size)

    model= selected_model(hidden_size_list=hidden_size_list, conv_layer_num=conv_layer_num, block_num=block_num, fc2_out=2, input_size=image_size)
    #optimizer 선택
    optimizer= optim.Adam(model.parameters(), lr=learning_rate)
    criterion= nn.CrossEntropyLoss()

    #모델 생성
    #모델 학습
    model= train_model(model=model,device=device, train_loader=train_loader, val_loader=val_loader, criterion=criterion, optimizer=optimizer, epochs=epochs)
    
    #모델평가
    accuracy, conf_matrix= evaluate_model(model=model, device=device, test_loader=test_loader)
    log_confusion_matrix(conf_matrix)
    wandb.log({"accuracy": accuracy})
    wandb.finish()

In [35]:
# params = {
#     "learning_rate": [0.001, 0.005, 0.01],
#     "image_size": [64, 128],
#     "batch_size": [128, 256],
#     "epochs": [ 20, 30],
#     "hidden_sizes": [[32,64,128],[64,128,256],[128,256,512]],
# }
params = {
    "learning_rate": [0.001],
    "image_size": [256],
    "batch_size": [128],
    "epochs": [20],
    "hidden_sizes": [[64,512]],
    "block_num": [4],
    "conv_layer_num": [2],
    
}
#hidden_sze가 계속 증가하는 이유 :  더 민감하게 반응하도록 하기 위해.


In [23]:

process_model(
    selected_model=ResidualConvolution,
    learning_rate=0.001,
    image_size=256,
    batch_size=128,
    epochs=20,
    block_num=4,
    conv_layer_num=2,
    hidden_size_list=[64,512],
)

Starting Epoch: 1...
Starting Epoch: 2...
Starting Epoch: 3...
Starting Epoch: 4...
Starting Epoch: 5...
Starting Epoch: 6...
Starting Epoch: 7...
Starting Epoch: 8...
Starting Epoch: 9...
Starting Epoch: 10...
Starting Epoch: 11...
Starting Epoch: 12...
Starting Epoch: 13...
Starting Epoch: 14...
Starting Epoch: 15...
Starting Epoch: 16...
Starting Epoch: 17...
Starting Epoch: 18...
Starting Epoch: 19...
Starting Epoch: 20...


accuracy,▁
accuracy,62.21563


In [24]:
class ResidualXavierConvBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super(ResidualXavierConvBlock, self).__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, 3, padding=1)
        self.conv2 = nn.Conv2d(out_channels, out_channels, 3, padding=1)
        
        # 스킵 커넥션을 위한 1x1 Conv (채널 수가 다를 때)
        self.shortcut = nn.Conv2d(in_channels, out_channels, 1) if in_channels != out_channels else nn.Identity()
        
        nn.init.xavier_normal_(self.conv1.weight)
        nn.init.xavier_normal_(self.conv2.weight)
        if isinstance(self.shortcut, nn.Conv2d):
            nn.init.xavier_normal_(self.shortcut.weight)
        
    def forward(self, x):
        residual = self.shortcut(x)
        
        out = F.relu(self.conv1(x))
        out = self.conv2(out)
        out = F.relu(out)
        out = out + residual
        return out


# 잔차 연결이 포함된 메인 모델
class ResidualConvolutionXavier(nn.Module):
    def __init__(self, hidden_size_list, conv_layer_num, block_num, fc2_out, input_size):
        super(ResidualConvolutionXavier, self).__init__()
        self.conv_layer_num = conv_layer_num
        self.block_num = block_num
        self.input_size = input_size
        self.start_size= hidden_size_list[0]
        self.max_size= math.log2(hidden_size_list[-1]/hidden_size_list[0])
        # Block들을 동적으로 생성
        self.blocks = nn.ModuleList()
        
        # 각 Block당 Conv 레이어 수 계산
        conv_per_block = conv_layer_num // block_num
        
        in_channels = 3
        for block_idx in range(block_num):
            block_layers = nn.ModuleList()
            
            # 각 Block 내부의 Residual Conv 레이어들
            for conv_idx in range(conv_per_block):
                global_conv_idx = block_idx * conv_per_block + conv_idx
                conv_size = global_conv_idx if global_conv_idx < self.max_size else self.max_size
                
                output_size = self.start_size * (2 ** conv_size) 
                
                block_layers.append(ResidualXavierConvBlock(in_channels, output_size))
                in_channels = output_size
            
            # Block 끝에 Pooling 추가
            block_layers.append(nn.MaxPool2d(2, 2))
            self.blocks.append(block_layers)
        
        # FC 레이어 크기 계산
        self.fc1_size = hidden_size_list[-1]
        self.fc2_size = fc2_out
        
        # FC 레이어는 forward에서 동적으로 생성
        self.fc1 = None
        self.fc2 = nn.Linear(self.fc1_size, fc2_out)
    
    def forward(self, x):
        # Block들을 통과
        for block in self.blocks:
            for layer in block:
                if isinstance(layer, ResidualXavierConvBlock):
                    x = layer(x)
                else:  # MaxPool2d
                    x = layer(x)
        
        # Flatten
        batch_size = x.size(0)
        features = x.size(1) * x.size(2) * x.size(3)
        x = x.view(batch_size, -1)
        
        # FC1 레이어 동적 생성
        if self.fc1 is None:
            self.fc1 = nn.Linear(features, self.fc1_size).to(x.device)
        
        # FC 레이어들 통과
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x

In [25]:

process_model(
    selected_model=ResidualConvolutionXavier,
    learning_rate=0.001,
    image_size=256,
    batch_size=128,
    epochs=20,
    block_num=4,
    conv_layer_num=2,
    hidden_size_list=[64,512],
)

Starting Epoch: 1...
Starting Epoch: 2...
Starting Epoch: 3...
Starting Epoch: 4...
Starting Epoch: 5...
Starting Epoch: 6...
Starting Epoch: 7...
Starting Epoch: 8...
Starting Epoch: 9...
Starting Epoch: 10...
Starting Epoch: 11...
Starting Epoch: 12...
Starting Epoch: 13...
Starting Epoch: 14...
Starting Epoch: 15...
Starting Epoch: 16...
Starting Epoch: 17...
Starting Epoch: 18...
Starting Epoch: 19...
Starting Epoch: 20...


accuracy,▁
accuracy,62.41345


In [37]:
class ResidualXavierBatchNormConvBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super(ResidualXavierBatchNormConvBlock, self).__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, 3, padding=1)
        self.conv2 = nn.Conv2d(out_channels, out_channels, 3, padding=1)
        # num_features: 채널 수 (conv의 출력 채널 수와 동일)
        # eps: 수치 안정성을 위한 작은 값 (기본값: 1e-5)
        # momentum: 이동평균 업데이트 비율 (기본값: 0.1)
        # affine: 학습 가능한 γ, β 파라미터 사용 여부 (기본값: True)
        # track_running_stats: 평균과 분산을 추적할지 여부 (기본값: True)
        self.bn1 = nn.BatchNorm2d(num_features=out_channels, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True) 
        self.bn2 = nn.BatchNorm2d(num_features=out_channels, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True) 

        # 스킵 커넥션을 위한 1x1 Conv (채널 수가 다를 때)
        self.shortcut = nn.Conv2d(in_channels, out_channels, 1) if in_channels != out_channels else nn.Identity()
        
        nn.init.xavier_normal_(self.conv1.weight)
        nn.init.xavier_normal_(self.conv2.weight)
        if isinstance(self.shortcut, nn.Conv2d):
            nn.init.xavier_normal_(self.shortcut.weight)
        
    def forward(self, x):
        residual = self.shortcut(x)
        
        out = F.relu(self.bn1(self.conv1(x)))
        out = F.relu(self.bn2(self.conv2(out)))
        out = out + residual
        return out


# 잔차 연결이 포함된 메인 모델
class ResidualConvolutionXavierBatchNorm(nn.Module):
    def __init__(self, hidden_size_list, conv_layer_num, block_num, fc2_out, input_size):
        super(ResidualConvolutionXavierBatchNorm, self).__init__()
        self.conv_layer_num = conv_layer_num
        self.block_num = block_num
        self.input_size = input_size
        self.start_size= hidden_size_list[0]
        self.max_size= math.log2(hidden_size_list[-1]/hidden_size_list[0])
        # Block들을 동적으로 생성
        self.blocks = nn.ModuleList()
        
        # 각 Block당 Conv 레이어 수 계산
        conv_per_block = conv_layer_num // block_num
        
        in_channels = 3
        for block_idx in range(block_num):
            block_layers = nn.ModuleList()
            
            # 각 Block 내부의 Residual Conv 레이어들
            for conv_idx in range(conv_per_block):
                global_conv_idx = block_idx * conv_per_block + conv_idx
                conv_size = global_conv_idx if global_conv_idx < self.max_size else self.max_size
                
                output_size = self.start_size * (2 ** conv_size) 
                
                block_layers.append(ResidualXavierConvBlock(in_channels, output_size))
                in_channels = output_size
            
            # Block 끝에 Pooling 추가
            block_layers.append(nn.MaxPool2d(2, 2))
            self.blocks.append(block_layers)
        
        # FC 레이어 크기 계산
        self.fc1_size = hidden_size_list[-1]
        self.fc2_size = fc2_out
        
        # FC 레이어는 forward에서 동적으로 생성
        self.fc1 = None
        self.fc2 = nn.Linear(self.fc1_size, fc2_out)
    
    def forward(self, x):
        # Block들을 통과
        for block in self.blocks:
            for layer in block:
                if isinstance(layer, ResidualXavierConvBlock):
                    x = layer(x)
                else:  # MaxPool2d
                    x = layer(x)
        
        # Flatten
        batch_size = x.size(0)
        features = x.size(1) * x.size(2) * x.size(3)
        x = x.view(batch_size, -1)
        
        # FC1 레이어 동적 생성
        if self.fc1 is None:
            self.fc1 = nn.Linear(features, self.fc1_size).to(x.device)
        
        # FC 레이어들 통과
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x

In [38]:

process_model(
    selected_model=ResidualConvolutionXavierBatchNorm,
    learning_rate=0.001,
    image_size=256,
    batch_size=128,
    epochs=20,
    block_num=4,
    conv_layer_num=2,
    hidden_size_list=[64,512],
)

Starting Epoch: 1...
Starting Epoch: 2...
Starting Epoch: 3...
Starting Epoch: 4...
Starting Epoch: 5...
Starting Epoch: 6...
Starting Epoch: 7...
Starting Epoch: 8...
Starting Epoch: 9...
Starting Epoch: 10...
Starting Epoch: 11...
Starting Epoch: 12...
Starting Epoch: 13...
Starting Epoch: 14...
Starting Epoch: 15...
Starting Epoch: 16...
Starting Epoch: 17...
Starting Epoch: 18...
Starting Epoch: 19...
Starting Epoch: 20...


accuracy,▁
accuracy,63.10584


In [ ]:
class ResidualXavierInstanceNormConvBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super(ResidualXavierInstanceNormConvBlock, self).__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, 3, padding=1)
        self.conv2 = nn.Conv2d(out_channels, out_channels, 3, padding=1)
        # num_features: 채널 수 (conv의 출력 채널 수와 동일)
        # eps: 수치 안정성을 위한 작은 값 (기본값: 1e-5)
        # momentum: 이동평균 업데이트 비율 (기본값: 0.1)
        # affine: 학습 가능한 γ, β 파라미터 사용 여부 (기본값: True)
        # track_running_stats: 평균과 분산을 추적할지 여부 (기본값: True)
        self.in1 = nn.InstanceNorm2d(num_features=out_channels, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        self.in2 = nn.InstanceNorm2d(num_features=out_channels, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)

        # 스킵 커넥션을 위한 1x1 Conv (채널 수가 다를 때)
        self.shortcut = nn.Conv2d(in_channels, out_channels, 1) if in_channels != out_channels else nn.Identity()
        
        nn.init.xavier_normal_(self.conv1.weight)
        nn.init.xavier_normal_(self.conv2.weight)
        if isinstance(self.shortcut, nn.Conv2d):
            nn.init.xavier_normal_(self.shortcut.weight)
        
    def forward(self, x):
        residual = self.shortcut(x)
        
        out = F.relu(self.in1(self.conv1(x)))
        out = F.relu(self.in2(self.conv2(out)))
        out = out + residual
        return out


# 잔차 연결이 포함된 메인 모델
class ResidualConvolutionXavierInstanceNorm(nn.Module):
    def __init__(self, hidden_size_list, conv_layer_num, block_num, fc2_out, input_size):
        super(ResidualConvolutionXavierInstanceNorm, self).__init__()
        self.conv_layer_num = conv_layer_num
        self.block_num = block_num
        self.input_size = input_size
        self.start_size= hidden_size_list[0]
        self.max_size= math.log2(hidden_size_list[-1]/hidden_size_list[0])
        # Block들을 동적으로 생성
        self.blocks = nn.ModuleList()
        
        # 각 Block당 Conv 레이어 수 계산
        conv_per_block = conv_layer_num // block_num
        
        in_channels = 3
        for block_idx in range(block_num):
            block_layers = nn.ModuleList()
            
            # 각 Block 내부의 Residual Conv 레이어들
            for conv_idx in range(conv_per_block):
                global_conv_idx = block_idx * conv_per_block + conv_idx
                conv_size = global_conv_idx if global_conv_idx < self.max_size else self.max_size
                
                output_size = self.start_size * (2 ** conv_size) 
                
                block_layers.append(ResidualXavierInstanceNormConvBlock(in_channels, output_size))
                in_channels = output_size
            
            # Block 끝에 Pooling 추가
            block_layers.append(nn.MaxPool2d(2, 2))
            self.blocks.append(block_layers)
        
        # FC 레이어 크기 계산
        self.fc1_size = hidden_size_list[-1]
        self.fc2_size = fc2_out
        
        # FC 레이어는 forward에서 동적으로 생성
        self.fc1 = None
        self.fc2 = nn.Linear(self.fc1_size, fc2_out)
    
    def forward(self, x):
        # Block들을 통과
        for block in self.blocks:
            for layer in block:
                if isinstance(layer, ResidualXavierInstanceNormConvBlock):
                    x = layer(x)
                else:  # MaxPool2d
                    x = layer(x)
        
        # Flatten
        batch_size = x.size(0)
        features = x.size(1) * x.size(2) * x.size(3)
        x = x.view(batch_size, -1)
        
        # FC1 레이어 동적 생성
        if self.fc1 is None:
            self.fc1 = nn.Linear(features, self.fc1_size).to(x.device)
        
        # FC 레이어들 통과
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x

In [40]:

process_model(
    selected_model=ResidualConvolutionXavierInstanceNorm,
    learning_rate=0.001,
    image_size=256,
    batch_size=128,
    epochs=20,
    block_num=4,
    conv_layer_num=2,
    hidden_size_list=[64,512],
)

Starting Epoch: 1...
Starting Epoch: 2...
Starting Epoch: 3...
Starting Epoch: 4...
Starting Epoch: 5...
Starting Epoch: 6...
Starting Epoch: 7...
Starting Epoch: 8...
Starting Epoch: 9...
Starting Epoch: 10...
Starting Epoch: 11...
Starting Epoch: 12...
Starting Epoch: 13...
Starting Epoch: 14...
Starting Epoch: 15...
Starting Epoch: 16...
Starting Epoch: 17...
Starting Epoch: 18...
Starting Epoch: 19...
Starting Epoch: 20...


accuracy,▁
accuracy,65.77646


$$
y = \frac{x - \mathrm{E}[x]}{\sqrt{\mathrm{Var}[x] + \epsilon}} \times \gamma + \beta
$$

| 구성 요소 | 의미 | 계산 범위 | 역할 |
|-----------|------|-----------|------|
| x | 입력 픽셀 값 | 개별 픽셀 | 원본 특성 |
| E[x] | 평균 | 한 이미지의 한 채널 내 모든 픽셀 | 중심화 |
| Var[x] | 분산 | 한 이미지의 한 채널 내 모든 픽셀 | 스케일링 |
| ε | 안정화 상수 | 스칼라 | 수치 안정성 |
| gamma | 스케일 파라미터 | 채널별 | 분산 조절 |
| beta | 시프트 파라미터 | 채널별 | 평균 조절 |

$$
y = \frac{x - \mathrm{E}[x]}{\sqrt{\mathrm{Var}[x] + \epsilon}} \times \gamma + \beta
$$

| 구성 요소 | 의미 | 계산 범위 | 역할 |
|-----------|------|-----------|------|
| x | 입력 픽셀 값 | 개별 픽셀 | 원본 특성 |
| E[x] | 평균 | 배치 내 모든 이미지의 같은 채널 내 모든 픽셀 | 중심화 |
| Var[x] | 분산 | 배치 내 모든 이미지의 같은 채널 내 모든 픽셀 | 스케일링 |
| ε | 안정화 상수 | 스칼라 | 수치 안정성 |
| gamma | 스케일 파라미터 | 채널별 | 분산 조절 |
| beta | 시프트 파라미터 | 채널별 | 평균 조절 |

In [ ]:
class ResidualXavierInstanceNormConvBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super(ResidualXavierInstanceNormConvBlock, self).__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, 3, padding=1)
        self.conv2 = nn.Conv2d(out_channels, out_channels, 3, padding=1)
        # num_features: 채널 수 (conv의 출력 채널 수와 동일)
        # eps: 수치 안정성을 위한 작은 값 (기본값: 1e-5)
        # momentum: 이동평균 업데이트 비율 (기본값: 0.1)
        # affine: 학습 가능한 γ, β 파라미터 사용 여부 (기본값: True)
        # track_running_stats: 평균과 분산을 추적할지 여부 (기본값: True)
        self.in1 = nn.InstanceNorm2d(num_features=out_channels, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        self.in2 = nn.InstanceNorm2d(num_features=out_channels, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)

        # 스킵 커넥션을 위한 1x1 Conv (채널 수가 다를 때)
        self.shortcut = nn.Conv2d(in_channels, out_channels, 1) if in_channels != out_channels else nn.Identity()
        
        nn.init.xavier_normal_(self.conv1.weight)
        nn.init.xavier_normal_(self.conv2.weight)
        if isinstance(self.shortcut, nn.Conv2d):
            nn.init.xavier_normal_(self.shortcut.weight)
        
    def forward(self, x):
        residual = self.shortcut(x)
        
        out = F.relu(self.in1(self.conv1(x)))
        out = F.relu(self.in2(self.conv2(out)))
        out = out + residual
        return out


# 잔차 연결이 포함된 메인 모델
class ResidualConvolutionXavierInstanceNorm(nn.Module):
    def __init__(self, hidden_size_list, conv_layer_num, block_num, fc2_out, input_size):
        super(ResidualConvolutionXavierInstanceNorm, self).__init__()
        self.conv_layer_num = conv_layer_num
        self.block_num = block_num
        self.input_size = input_size
        self.start_size= hidden_size_list[0]
        self.max_size= math.log2(hidden_size_list[-1]/hidden_size_list[0])
        # Block들을 동적으로 생성
        self.blocks = nn.ModuleList()
        
        # 각 Block당 Conv 레이어 수 계산
        conv_per_block = conv_layer_num // block_num
        
        in_channels = 3
        for block_idx in range(block_num):
            block_layers = nn.ModuleList()
            
            # 각 Block 내부의 Residual Conv 레이어들
            for conv_idx in range(conv_per_block):
                global_conv_idx = block_idx * conv_per_block + conv_idx
                conv_size = global_conv_idx if global_conv_idx < self.max_size else self.max_size
                
                output_size = self.start_size * (2 ** conv_size) 
                
                block_layers.append(ResidualXavierInstanceNormConvBlock(in_channels, output_size))
                in_channels = output_size
            
            # Block 끝에 Pooling 추가
            block_layers.append(nn.MaxPool2d(2, 2))
            self.blocks.append(block_layers)
        
        # FC 레이어 크기 계산
        self.fc1_size = hidden_size_list[-1]
        self.fc2_size = fc2_out
        
        # FC 레이어는 forward에서 동적으로 생성
        self.fc1 = None
        self.fc2 = nn.Linear(self.fc1_size, fc2_out)
    
    def forward(self, x):
        # Block들을 통과
        for block in self.blocks:
            for layer in block:
                if isinstance(layer, ResidualXavierInstanceNormConvBlock):
                    x = layer(x)
                else:  # MaxPool2d
                    x = layer(x)
        
        # Flatten
        batch_size = x.size(0)
        features = x.size(1) * x.size(2) * x.size(3)
        x = x.view(batch_size, -1)
        
        # FC1 레이어 동적 생성
        if self.fc1 is None:
            self.fc1 = nn.Linear(features, self.fc1_size).to(x.device)
        
        # FC 레이어들 통과
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x

In [ ]:

process_model(
    selected_model=ResidualConvolutionXavierInstanceNorm,
    learning_rate=0.001,
    image_size=256,
    batch_size=128,
    epochs=20,
    block_num=4,
    conv_layer_num=2,
    hidden_size_list=[64,512],
)

In [ ]:
import torchvision.transforms as transforms

# Translation Invariance 강화를 위한 변환
transform = transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.5),    # 좌우 반전
    transforms.RandomVerticalFlip(p=0.3),      # 상하 반전
    transforms.RandomRotation(degrees=15),     # 회전
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1)),  # 평행이동
    transforms.RandomCrop(64, padding=4),      # 자르기
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], 
                       std=[0.229, 0.224, 0.225])
])